# Shot Data Visualizations

This notebook starts with an interactive heatmap for shot efficiency by distance-based shot location. The processed dataset does not include x/y court coordinates, so location is represented by shot zone and shot-distance bins.

The primary metric is expected shot value: model-predicted make probability multiplied by the shot point value.

In [6]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import Dropdown, IntSlider, SelectMultiple, interact
import matplotlib.patches as patches
from scipy.stats import binned_statistic_2d
import plotly.graph_objects as go

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

In [7]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data/processed/shot_logs_cleaned_engineered.csv"

df = pd.read_csv(DATA_PATH)
df["actual_points"] = df["made"] * df["pts_type"]

df.shape, df.head()

((127757, 29),
      player_name                   matchup location  period  game_clock_sec  \
 0  brian roberts  MAR 04, 2015 - CHA @ BKN        A       1              69   
 1  brian roberts  MAR 04, 2015 - CHA @ BKN        A       1              14   
 2  brian roberts  MAR 04, 2015 - CHA @ BKN        A       1               0   
 3  brian roberts  MAR 04, 2015 - CHA @ BKN        A       2             707   
 4  brian roberts  MAR 04, 2015 - CHA @ BKN        A       2             634   
 
    shot_clock  shot_clock_missing  dribbles  touch_time  shot_dist  ...  \
 0        10.8                   0         2         1.9        7.7  ...   
 1         3.4                   0         0         0.8       28.2  ...   
 2         0.0                   1         3         2.7       10.1  ...   
 3        10.3                   0         2         1.9       17.2  ...   
 4        10.9                   0         2         2.7        3.7  ...   
 
    three_pointer  paint  mid_range  pull_up 

## Bin Definition

In [8]:
# Use actual point value from dataset to define 2PT vs 3PT
zone_conditions = [
    (df["pts_type"].eq(2)) & (df["paint"].eq(1)),
    (df["pts_type"].eq(2)) & (df["mid_range"].eq(1)),
    df["pts_type"].eq(3),
]

zone_labels = ["Paint", "Mid-range", "3PT"]

df["shot_zone"] = np.select(zone_conditions, zone_labels, default="Other")

# copy original distances for visualization binning only
df["shot_dist_for_bins"] = df["shot_dist"]

# Fix sub-22 recorded threes -> treat as corner threes
df.loc[
    (df["pts_type"] == 3) &
    (df["shot_dist_for_bins"] < 22),
    "shot_dist_for_bins"
] = 22

# Fix 24+ recorded twos -> treat as long twos
df.loc[
    (df["pts_type"] == 2) &
    (df["shot_dist_for_bins"] >= 24),
    "shot_dist_for_bins"
] = 23.9


# Distance bins by zone:
distance_bins = [0, 4, 8, 12, 16, 20, 22, 24, 28, 32, 40, np.inf]

distance_labels = [
    "0-4",
    "4-8",
    "8-12",
    "12-16",
    "16-20",
    "20-22",
    "22-24",
    "24-28",
    "28-32",
    "32-40",
    "40+"
]

df["shot_distance_bin"] = pd.cut(
    df["shot_dist_for_bins"],
    bins=distance_bins,
    labels=distance_labels,
    include_lowest=True,
    right=False,
)


pressure_bins = [-0.01, 2, 4, 6, np.inf]
pressure_labels = [
    "Very tight (0-2 ft)",
    "Tight (2-4 ft)",
    "Moderate (4-6 ft)",
    "Open (6+ ft)"
]

df["def_pressure_bin"] = pd.cut(
    df["close_def_dist"],
    bins=pressure_bins,
    labels=pressure_labels,
)


dribble_bins = [-0.01, 0, 2, 6, np.inf]
dribble_labels = [
    "0 dribbles",
    "1-2 dribbles",
    "3-6 dribbles",
    "7+ dribbles"
]

df["dribble_bin"] = pd.cut(
    df["dribbles"],
    bins=dribble_bins,
    labels=dribble_labels,
    include_lowest=True,
)


df[[
    "pts_type",
    "shot_zone",
    "shot_distance_bin",
    "def_pressure_bin",
    "dribble_bin",
    "actual_points"
]].head()

,pts_type,shot_zone,shot_distance_bin,def_pressure_bin,dribble_bin,actual_points
0,2,Paint,4-8,Very tight (0-2 ft),1-2 dribbles,2
1,3,3PT,28-32,Open (6+ ft),0 dribbles,0
2,2,Mid-range,8-12,Very tight (0-2 ft),3-6 dribbles,0
3,2,Mid-range,16-20,Tight (2-4 ft),1-2 dribbles,0
4,2,Paint,0-4,Very tight (0-2 ft),1-2 dribbles,0


## Estimate Expected Shot Value

Expected shot value is calculated as:

`predicted make probability * pts_type`

This loads the fitted XGBoost model saved by notebook 07, then uses its predicted make probability for every shot.

In [9]:
MODEL_PATH = PROJECT_ROOT / "models/xgboost_engineered_shot_model.joblib"

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {MODEL_PATH}. Run notebook 07 through the model-saving cell first."
    )

model_bundle = joblib.load(MODEL_PATH)
shot_value_model = model_bundle["model"]
model_features = model_bundle["feature_columns"]


def build_model_matrix(data, feature_columns):
    X_model = data.copy()
    if "location" in feature_columns:
        X_model["location"] = (data["location"] == "H").astype(int)

    missing_features = [col for col in feature_columns if col not in X_model.columns]
    if missing_features:
        raise ValueError(f"Missing model features: {missing_features}")

    return X_model[feature_columns]


X = build_model_matrix(df, model_features)
df["make_probability"] = shot_value_model.predict_proba(X)[:, 1]
df["expected_shot_value"] = df["make_probability"] * df["pts_type"]

print(f"Loaded XGBoost model from {MODEL_PATH}")
print(f"Using {len(model_features)} features from notebook 07")

df[["shot_dist", "pts_type", "make_probability", "expected_shot_value", "actual_points"]].head()

Loaded XGBoost model from c:\Users\micha\Downloads\CS158\Shot-Value-Machine-Learning\models\xgboost_engineered_shot_model.joblib
Using 22 features from notebook 07


,shot_dist,pts_type,make_probability,expected_shot_value,actual_points
0,7.7,2,0.370974,0.741948,2
1,28.2,3,0.217277,0.651830,0
2,10.1,2,0.069885,0.139769,0
3,17.2,2,0.391796,0.783592,0
4,3.7,2,0.431299,0.862598,0


## Binned Expected Value by Defender Distance

In [11]:
pressure_labels = [
    "Very tight (0-2 ft)",
    "Tight (2-4 ft)",
    "Moderate (4-6 ft)",
    "Open (6+ ft)"
]

pressure_ev = (
    df.dropna(
        subset=[
            "shot_distance_bin",
            "def_pressure_bin",
            "expected_shot_value"
        ]
    )
    .groupby(
        ["shot_distance_bin","def_pressure_bin"],
        observed=False
    )
    .agg(
        expected_value=("expected_shot_value","mean"),
        actual_points_per_shot=("actual_points","mean"),
        attempts=("expected_shot_value","size"),
    )
    .reset_index()
)

min_attempts = 75
pressure_plot = pressure_ev[
    pressure_ev["attempts"] >= min_attempts
].copy()

fig = go.Figure()

colors = {
    "Very tight (0-2 ft)": "#1f77b4",
    "Tight (2-4 ft)": "#ff7f0e",
    "Moderate (4-6 ft)": "#2ca02c",
    "Open (6+ ft)": "#d62728",
}

for label in pressure_labels:

    line_data = (
        pressure_plot[
            pressure_plot["def_pressure_bin"] == label
        ]
        .set_index("shot_distance_bin")
        .reindex(distance_labels)
        .reset_index()
    )

    fig.add_trace(
        go.Scatter(
            x=line_data["shot_distance_bin"],
            y=line_data["expected_value"],
            mode="lines+markers",
            name=label,

            line=dict(
                width=3,
                color=colors[label]
            ),

            marker=dict(
                size=8,
                color=colors[label]
            ),

            customdata=np.column_stack([
                line_data["attempts"].fillna(0),
                line_data["actual_points_per_shot"]
            ]),

            hovertemplate=
                "<b>%{x}</b><br>" +
                "Pressure: " + label + "<br>" +
                "Expected Value: %{y:.3f}<br>" +
                "Actual Pts/Shot: %{customdata[1]:.3f}<br>" +
                "Attempts: %{customdata[0]:,.0f}" +
                "<extra></extra>"
        )
    )


league_avg = df["expected_shot_value"].mean()

if "22-24 (Corner 3)" in distance_labels:
    split = distance_labels.index("22-24 (Corner 3)")
    fig.add_vline(
        x=split - 0.5,
        line_dash="dot",
        line_color="black",
        annotation_text="3PT range",
        annotation_position="top"
    )


fig.update_layout(
    title=dict(
        text="Shot Value vs Distance Under Different Defensive Pressure",
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    ),

    xaxis_title="Shot Distance",
    yaxis_title="Expected Points per Shot",

    template="simple_white",
    width=1000,
    height=575,

    hovermode="closest",

    legend=dict(
        title="Defensive Pressure",
        x=0.72,
        y=0.98,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.75)",
        bordercolor="rgba(0,0,0,0.15)",
        borderwidth=1
    ),

    margin=dict(
        l=70,
        r=40,
        t=80,
        b=80
    )
)

fig.update_xaxes(
    tickangle=35,
    showgrid=True,
    gridcolor="rgba(0,0,0,0.12)",
    range=[-0.5, 7.5]   # removes blank 32-40 and 40+ space
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.18)"
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': {'bdata': ('AAAAAIBoxUCI6SEUAvfwPwAAAAAAZ8' ... 'AAAAD4fwAAAAAAAAAAAAAAAAAA+H8='),
                             'dtype': 'f8',
                             'shape': '11, 2'},
              'hovertemplate': ('<b>%{x}</b><br>Pressure: Very ' ... 'omdata[0]:,.0f}<extra></extra>'),
              'line': {'color': '#1f77b4', 'width': 3},
              'marker': {'color': '#1f77b4', 'size': 8},
              'mode': 'lines+markers',
              'name': 'Very tight (0-2 ft)',
              'type': 'scatter',
              'x': array(['0-4', '4-8', '8-12', '12-16', '16-20', '20-22', '22-24', '24-28',
                          '28-32', '32-40', '40+'], dtype=object),
              'y': {'bdata': ('gnbOmPjM8D9zxQ9u9LHqP5GvAL1iTe' ... 'AAAPh/AAAAAAAA+H8AAAAAAAD4fw=='),
                    'dtype': 'f8'}},
             {'customdata': {'bdata': ('AAAAAADWxUAO7+ox3XP1PwAAAACA2c' ... 'AAAAD4fwAAAAAAAAAAAAAAAAAA+H8='),
                             'dtype': 'f8',
                             'shape': '11, 2'},
              'hovertemplate': ('<b>%{x}</b><br>Pressure: Tight' ... 'omdata[0]:,.0f}<extra></extra>'),
              'line': {'color': '#ff7f0e', 'width': 3},
              'marker': {'color': '#ff7f0e', 'size': 8},
              'mode': 'lines+markers',
              'name': 'Tight (2-4 ft)',
              'type': 'scatter',
              'x': array(['0-4', '4-8', '8-12', '12-16', '16-20', '20-22', '22-24', '24-28',
                          '28-32', '32-40', '40+'], dtype=object),
              'y': {'bdata': ('nI4BZbEQ9T+jELOde3zvP22CqOb65u' ... '3k4N8/AAAAAAAA+H8AAAAAAAD4fw=='),
                    'dtype': 'f8'}},
             {'customdata': {'bdata': ('AAAAAAD+o0Dm+Lg5QYz6PwAAAAAAqJ' ... 'kN5TXUPwAAAAAAAAAAAAAAAAAA+H8='),
                             'dtype': 'f8',
                             'shape': '11, 2'},
              'hovertemplate': ('<b>%{x}</b><br>Pressure: Moder' ... 'omdata[0]:,.0f}<extra></extra>'),
              'line': {'color': '#2ca02c', 'width': 3},
              'marker': {'color': '#2ca02c', 'size': 8},
              'mode': 'lines+markers',
              'name': 'Moderate (4-6 ft)',
              'type': 'scatter',
              'x': array(['0-4', '4-8', '8-12', '12-16', '16-20', '20-22', '22-24', '24-28',
                          '28-32', '32-40', '40+'], dtype=object),
              'y': {'bdata': ('EJ82AsUq+j+/Fk6zskr0P0hiDfbYQ+' ... 'GUwuU/Q3kNkERn1D8AAAAAAAD4fw=='),
                    'dtype': 'f8'}},
             {'customdata': {'bdata': ('AAAAAAAIkUAP4MN/8AD+PwAAAAAAwH' ... 'BPCfLUPwAAAAAAAAAAAAAAAAAA+H8='),
                             'dtype': 'f8',
                             'shape': '11, 2'},
              'hovertemplate': ('<b>%{x}</b><br>Pressure: Open ' ... 'omdata[0]:,.0f}<extra></extra>'),
              'line': {'color': '#d62728', 'width': 3},
              'marker': {'color': '#d62728', 'size': 8},
              'mode': 'lines+markers',
              'name': 'Open (6+ ft)',
              'type': 'scatter',
              'x': array(['0-4', '4-8', '8-12', '12-16', '16-20', '20-22', '22-24', '24-28',
                          '28-32', '32-40', '40+'], dtype=object),
              'y': {'bdata': ('CqrW98ui/T/Y6/WC1w36PyIiInLXHP' ... 'Tu7OQ/gjwlBNb/2j8AAAAAAAD4fw=='),
                    'dtype': 'f8'}}],
    'layout': {'height': 575,
               'hovermode': 'closest',
               'legend': {'bgcolor': 'rgba(255,255,255,0.75)',
                          'bordercolor': 'rgba(0,0,0,0.15)',
                          'borderwidth': 1,
                          'title': {'text': 'Defensive Pressure'},
                          'x': 0.72,
                          'xanchor': 'left',
                          'y': 0.98,
                          'yanchor': 'top'},
               'margin': {'b': 80, 'l': 70, 'r': 40, 't': 80},
               'template': '...',
               'title': {'font': {'size': 20},
                         'text': 'Shot Value vs 

## Interactive Heatmap

In [12]:
metric_options = {
    "Expected value": "expected_shot_value",
    "Actual points per shot": "actual_points",
    "Field goal percentage": "made",
}

zone_order = ["Paint", "Mid-range", "3PT", "Other"]
distance_order = distance_labels

pressure_selector = SelectMultiple(
    options=pressure_labels,
    value=tuple(pressure_labels),
    description="Pressure",
    rows=len(pressure_labels),
)

dribble_selector = SelectMultiple(
    options=dribble_labels,
    value=tuple(dribble_labels),
    description="Dribbles",
    rows=len(dribble_labels),
)

metric_selector = Dropdown(
    options=list(metric_options.keys()),
    value="Expected value",
    description="Metric",
)

min_attempts_slider = IntSlider(
    value=75,
    min=10,
    max=500,
    step=5,
    description="Min shots",
)


def plot_shot_value_heatmap(pressure_ranges, dribble_ranges, metric_name, min_attempts):
    metric_col = metric_options[metric_name]

    filtered = df[
        df["def_pressure_bin"].isin(pressure_ranges)
        & df["dribble_bin"].isin(dribble_ranges)
        & df[metric_col].notna()
    ].copy()

    if filtered.empty:
        print("No shots match the selected filters.")
        return

    grouped = (
        filtered.groupby(["shot_zone", "shot_distance_bin"], observed=False)
        .agg(value=(metric_col, "mean"), attempts=(metric_col, "size"))
        .reset_index()
    )

    value_grid = grouped.pivot(
        index="shot_zone",
        columns="shot_distance_bin",
        values="value"
    ).reindex(index=zone_order, columns=distance_order)

    attempts_grid = grouped.pivot(
        index="shot_zone",
        columns="shot_distance_bin",
        values="attempts"
    ).reindex(index=zone_order, columns=distance_order).fillna(0)

    display_grid = value_grid.mask(attempts_grid < min_attempts)

    # remove fully empty rows/columns
    display_grid = display_grid.dropna(axis=0, how="all")
    display_grid = display_grid.dropna(axis=1, how="all")

    attempts_grid = attempts_grid.loc[
        display_grid.index,
        display_grid.columns
    ]

    fig, ax = plt.subplots(figsize=(11, 4.5))

    avg_value = filtered[metric_col].mean()

    im = ax.imshow(
        display_grid,
        cmap="RdYlGn",
        aspect="auto",
        vmin=avg_value - 0.35,
        vmax=avg_value + 0.35
    )

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Points per shot")

    ax.set_xticks(np.arange(len(display_grid.columns)))
    ax.set_xticklabels(display_grid.columns)

    ax.set_yticks(np.arange(len(display_grid.index)))
    ax.set_yticklabels(display_grid.index)

    ax.set_xlabel("Shot distance bin (feet)")
    ax.set_ylabel("Shot zone")

    ax.set_title(
        f"{metric_name} by Zone and Distance\n"
        f"{len(filtered):,} shots selected | white/blank = fewer than {min_attempts} shots",
        pad=12
    )

    for y_idx, zone in enumerate(display_grid.index):
        for x_idx, distance in enumerate(display_grid.columns):
            value = display_grid.loc[zone, distance]
            attempts = attempts_grid.loc[zone, distance]

            if pd.notna(value):
                ax.text(
                    x_idx,
                    y_idx,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=9,
                    fontweight="bold",
                    color="black"
                )

                ax.text(
                    x_idx,
                    y_idx + 0.25,
                    f"n={int(attempts):,}",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="black"
                )

    # reduce visual clutter
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
interact(
    plot_shot_value_heatmap,
    pressure_ranges=pressure_selector,
    dribble_ranges=dribble_selector,
    metric_name=metric_selector,
    min_attempts=min_attempts_slider,
);

interactive(children=(SelectMultiple(description='Pressure', index=(0, 1, 2, 3), options=('Very tight (0-2 ft)…

: 